# Time-Series Model Forecasting Agent Platform Pipelines

**Learning Objectives:**
1. Learn how to use Agent Platform AutoML pre-built components
1. Learn how to build the Agent Platform AutoML pipeline with these components using BigQuery as a data source
1. Learn how to compile, upload, and run the Agent Platform AutoML pipeline


In this lab, you will build, deploy, and run the Agent Platform AutoML pipeline that orchestrates the **Agent Platform AutoML AI** services to train, tune, and serve batch predictions to BigQuery with a model. 

## Setup

In [ ]:
import os

from google.cloud import aiplatform

%load_ext google.cloud.bigquery

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value project)
PROJECT = PROJECT[0]

os.environ["PROJECT"] = PROJECT

### Create BigQuery View for prediction input

If you have not gone through the KFP Walkthrough lab, you will need to run the following cell to create a BigQuery dataset and table containing the data required for this lab.

In [ ]:
%env PROJECT = {PROJECT}
%env REGION = "us-central1"

%load_ext google.cloud.bigquery

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
CREATE OR REPLACE VIEW `power_grid.v_input_prep` AS
WITH max_time_anchor AS (
  -- Find the absolute latest timestamp in your historical raw data
  SELECT MAX(timestamp) AS global_max_timestamp
  FROM `power_grid.grid_load_train_demo`
),

historical_series AS (
  -- Retrieve only the last 168 hours (7 days) of actual data relative to the maximum timestamp
  SELECT
    t.timestamp,
    t.line_id,
    t.feeder_id,
    t.substation_id,
    t.grid_level,
    t.substation_type,
    t.load_mw,
    t.design_capacity_mw,
    t.temperature_c,
    t.is_weekend
  FROM `power_grid.grid_load_train_demo` t
  CROSS JOIN max_time_anchor m
  WHERE t.timestamp >= TIMESTAMP_SUB(m.global_max_timestamp, INTERVAL 168 HOUR)
),
  
latest_state AS (
  SELECT
    line_id,
    feeder_id,
    substation_id,
    grid_level,
    substation_type,
    design_capacity_mw,
    MAX(timestamp) AS last_timestamp
  FROM historical_series
  GROUP BY 1, 2, 3, 4, 5, 6
),
  
-- Generate the 24 hourly horizon timestamps starting right after the latest historical row
future_horizon_timestamps AS (
  SELECT
    ls.line_id,
    ls.feeder_id,
    ls.substation_id,
    ls.grid_level,
    ls.substation_type,
    ls.design_capacity_mw,
    future_ts AS timestamp
  FROM latest_state ls
  CROSS JOIN UNNEST(
    GENERATE_TIMESTAMP_ARRAY(
      TIMESTAMP_ADD(ls.last_timestamp, INTERVAL 1 HOUR),
      TIMESTAMP_ADD(ls.last_timestamp, INTERVAL 24 HOUR),
      INTERVAL 1 HOUR
    )
  ) AS future_ts
),
  
-- Construct placeholder rows with NULL targets and future covariates
future_placeholder_rows AS (
  SELECT
    f.timestamp,
    f.line_id,
    f.feeder_id,
    f.substation_id,
    f.grid_level,
    f.substation_type,
    CAST(NULL AS FLOAT64) AS load_mw,  -- Explicit NULL target for forecasting 
    f.design_capacity_mw,
    24.0 AS temperature_c,
    -- Dynamic check for weekend (Saturday=7, Sunday=1 in BigQuery EXTRACT DAYOFWEEK)
    CASE 
      WHEN EXTRACT(DAYOFWEEK FROM f.timestamp) IN (1, 7) THEN '1' 
      ELSE '0' 
    END AS is_weekend
  FROM future_horizon_timestamps f
)
  
-- Consolidated dataset: historical actuals (last 168h) + future horizon placeholders (next 24h)
SELECT * FROM historical_series
UNION ALL
SELECT * FROM future_placeholder_rows
ORDER BY line_id, timestamp ASC;

## Understanding the pipeline design


The workflow implemented by the pipeline is defined using a Python based Domain Specific Language (DSL). The pipeline's DSL is in the `pipeline/pipeline_automl_batch_preds.py` file that we will generate below.

The pipeline's DSL has been designed to avoid hardcoding any environment specific settings like file paths or connection strings. These settings are provided to the pipeline code through a set of environment variables.


## Building and deploying the pipeline

Let us write the pipeline to disk:

In [ ]:
%%writefile ./pipeline/power_grid_forecast_batch_job.py
"""Kubeflow Power-Grid Pipeline."""

import os
from google_cloud_pipeline_components.v1.bigquery import BigqueryQueryJobOp

from google_cloud_pipeline_components.v1.automl.training_job import (
    AutoMLTabularTrainingJobRunOp, AutoMLForecastingTrainingJobRunOp
)

from google_cloud_pipeline_components.types import artifact_types
from google_cloud_pipeline_components.v1.model import ModelGetOp
from google_cloud_pipeline_components.v1.batch_predict_job import (
    ModelBatchPredictOp
)

from kfp import dsl

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
INPUT_TABLE = 'forecasting_tmp_input'
#FORECAST_TABLE = 'forecasting_tmp_input'
INPUT_BASE_VIEW = 'v_input_dataset_prep'
BQ_DATASET = 'power_grid'
PROJECT = os.getenv("PROJECT")
REGION = os.getenv("REGION")
DATASET_SOURCE = os.getenv("DATASET_SOURCE")
PIPELINE_NAME = os.getenv("PIPELINE_NAME", "power-grid")
DISPLAY_NAME = os.getenv("MODEL_DISPLAY_NAME", PIPELINE_NAME)
#TARGET_COLUMN = os.getenv("TARGET_COLUMN", "Cover_Type")
BATCH_PREDS_SOURCE_URI = os.getenv("BATCH_PREDS_SOURCE_URI")
# Agent Platform AutoML Component image version
KFP_IMAGE = "gcr.io/ml-pipeline/google-cloud-pipeline-components:2.16.1"
MODEL_ID = os.getenv("MODEL_ID") #like '9133397351086424064'

from kfp.dsl import Input, Artifact

# Create a lightweight helper component
@dsl.component
def process_predictions_op(
    project_id: str,
    # Use the general Artifact class or google.VertexBatchPredictionJob if imported
    batch_job: Input[Artifact]
) -> str:
    # Access the property from the metadata dictionary
    bq_table = batch_job.metadata.get('bigqueryOutputTable')
    bq_dataset = batch_job.metadata.get('bigqueryOutputDataset') # e.g., bq://project.power_grid
    bq_dataset = bq_dataset.replace("bq://","")

    print(f"BigQuery table with predictions: {bq_table} in dataset {bq_dataset}")
    # ... your processing sql code ...
    postprocessing_sql = f"""
    SELECT
      design_capacity_mw,
      feeder_id,
      grid_level,
      is_weekend,
      line_id,
      predicted_load_mw.value AS load_mw,
      '1' AS is_forcast,
      substation_id,
      substation_type,
      temperature_c,
      CAST(timestamp AS TIMESTAMP) AS timestamp
    FROM
      `{bq_dataset}.{bq_table}`
    UNION ALL
    SELECT
      design_capacity_mw,
      feeder_id,
      grid_level,
      is_weekend,
      line_id,
      load_mw,
      '0' AS is_forcast,
      substation_id,
      substation_type,
      temperature_c,
      timestamp
    FROM
      `{bq_dataset}.forecasting_tmp_input`
    ORDER BY
      line_id,
      timestamp ASC
    """
    return postprocessing_sql

@dsl.pipeline(
    name=f"{PIPELINE_NAME}-automl-pipeline",
    description=f"AutoML Pipeline for {PIPELINE_NAME}",
    pipeline_root=PIPELINE_ROOT
)
def create_pipeline(model_id: str):

    bq_query = f"""
    CREATE OR REPLACE TABLE `{BQ_DATASET}.{INPUT_TABLE}` AS
    SELECT * FROM `{BQ_DATASET}.{INPUT_BASE_VIEW}`
    """

    bq_update_dataset_op = BigqueryQueryJobOp(
        project=PROJECT,
        location="US",
        query=bq_query,
    )
    
    get_model_task = ModelGetOp(
        project=PROJECT,
        location=REGION,
        model_name=f"{model_id}@default"
    )
    get_model_task.after(bq_update_dataset_op)

    # BigQuery URIs (Note the bq:// prefix)
    bigquery_source_input_uri = f"bq://{PROJECT}.{BQ_DATASET}.{INPUT_TABLE}"
    bigquery_destination_output_uri = f"bq://{PROJECT}.{BQ_DATASET}"

    # Instantiate the Batch Prediction Operator configured for BigQuery
    batch_predict_task = ModelBatchPredictOp(
        project=PROJECT,
        location=REGION,
        job_display_name="power-greed-batch-forecast",
        model=get_model_task.outputs["model"],

        # Configure Input from BigQuery
        instances_format="bigquery",
        bigquery_source_input_uri=bigquery_source_input_uri,

        # Configure Output to BigQuery
        predictions_format="bigquery",
        bigquery_destination_output_uri=bigquery_destination_output_uri
    )

    # Pass the artifact itself to the downstream component
    process_predictions_prep = process_predictions_op(
        project_id=PROJECT,
        batch_job=batch_predict_task.outputs['batchpredictionjob']
    )

    # Run predictions post-processing step
    bq_process_predictions_op = BigqueryQueryJobOp(
        project=PROJECT,
        location="US",
        query=process_predictions_prep.output,
    )

Overwriting ./pipeline/pipeline_automl_batch_preds.py

### Compile the pipeline

Let's start by defining the environment variables that will be passed to the pipeline compiler:

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATASET_SOURCE = f"bq://{PROJECT}.power_grid.grid_load_demo"
BATCH_PREDS_SOURCE_URI = f"bq://{PROJECT}.power_grid.grid_load_forecast_demo"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT={PROJECT}
%env REGION={REGION}
%env DATASET_SOURCE={DATASET_SOURCE}
%env BATCH_PREDS_SOURCE_URI={BATCH_PREDS_SOURCE_URI}

Let us make sure that the `ARTIFACT_STORE` has been created, and let us create it if not:

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

#### Use the CLI compiler to compile the pipeline

We compile the pipeline from the Python file we generated into a YAML description using the following command:

In [ ]:
PIPELINE_YAML = "power_grid_forecast_batch_job.yaml"

In [ ]:
!kfp dsl compile --py pipeline/power_grid_forecast_batch_job.py --output $PIPELINE_YAML

/home/jupyter/asl-ml-immersion/notebooks/kubeflow_pipelines/pipelines/solutions/covertype_automl_pipeline_batch_preds.yaml

**Note:** You can also use the Python SDK to compile the pipeline:

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

The result is the pipeline file. 

In [ ]:
!head {PIPELINE_YAML}

### Deploy the pipeline package

In [ ]:
aiplatform.init(project=PROJECT, location=REGION)

#TODO: PUT ACTUAL MODEL ID
MODEL_ID='9133397351086424064'

# Define the parameters:
JOB_PARAMETERS = {
    "model_id": MODEL_ID
}

pipeline_job = aiplatform.PipelineJob(
    display_name="power_grid_forecast_batch_job",
    template_path=PIPELINE_YAML,
    parameter_values=JOB_PARAMETERS,
    enable_caching=False,
)

pipeline_job.run()
#pipeline_job.submit()

### Schedule retraining pipeline

In [ ]:
from google.cloud import aiplatform

# Wrap the job in a Schedule object
pipeline_schedule = aiplatform.PipelineJobSchedule(
    pipeline_job=pipeline_job,
    display_name="hourly-batch-forecast-schedule-demo"
)

# TODO: Uncomment to start the schedule using a Cron string
# pipeline_schedule.create(
#     cron="0 * * * *",              # Runs exactly at the top of every hour
#     max_concurrent_run_count=1     # Prevents 2 pipelines from running at once if one takes > 60 mins
# )
#print(f"Schedule created! View it at: {pipeline_schedule.resource_name}")

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.